In [6]:
# Cell 1. 환경 연결 + 테이블 초기화 + 정상 데이터 적재
from pyspark.sql import SparkSession

BUCKET = "s3-meta-iceberg-test"
WAREHOUSE = f"s3://{BUCKET}/lakehouse-lab/warehouse"
DB = "ad_lakehouse"
TABLE = "maintenance_lab_events"
FULL_TABLE = f"glue_catalog.{DB}.{TABLE}"
S3_TABLE_PATH = f"s3://{BUCKET}/lakehouse-lab/warehouse/{DB}.db/{TABLE}"

spark = (
    SparkSession.builder
    .appName("Iceberg-S3-Maintenance-Lab-Minimal")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2,"
            "org.apache.iceberg:iceberg-aws-bundle:1.9.2")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.glue_catalog.warehouse", WAREHOUSE)
    .config("spark.sql.catalog.glue_catalog.glue.region", "ap-northeast-2")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.default.parallelism", "2")
    .getOrCreate()
)

print("Spark 연결 완료:", spark.version)
print("Warehouse:", WAREHOUSE)

spark.sql(f"CREATE DATABASE IF NOT EXISTS glue_catalog.{DB}")
spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE}")
spark.sql(f"""
CREATE TABLE {FULL_TABLE} (
    event_id STRING,
    event_ts TIMESTAMP,
    user_id STRING,
    ad_id STRING,
    event_type STRING,
    cost DOUBLE,
    event_date DATE
)
USING iceberg
PARTITIONED BY (event_date)
TBLPROPERTIES (
    'format-version' = '2',
    'write.target-file-size-bytes' = '134217728'
)
""")

spark.sql(f"""
INSERT INTO {FULL_TABLE} VALUES
('imp_1001',  TIMESTAMP '2026-05-03 09:00:00', 'user_A', 'ad_10', 'impression', 0.12, DATE '2026-05-03'),
('clk_1001',  TIMESTAMP '2026-05-03 09:03:00', 'user_A', 'ad_10', 'click',      0.00, DATE '2026-05-03'),
('conv_1001', TIMESTAMP '2026-05-03 09:20:00', 'user_A', 'ad_10', 'conversion', 0.00, DATE '2026-05-03')
""")

print("[테이블 확인]")
spark.sql(f"SHOW TABLES IN glue_catalog.{DB}").show(truncate=False)

print("[정상 데이터 확인]")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY event_ts").show(truncate=False)

print("[현재 snapshot 목록]")
spark.sql(f"""
SELECT committed_at, snapshot_id, operation
FROM {FULL_TABLE}.snapshots
ORDER BY committed_at
""").show(truncate=False)

Spark 연결 완료: 3.5.1
Warehouse: s3://s3-meta-iceberg-test/lakehouse-lab/warehouse
[테이블 확인]
+------------+----------------------+-----------+
|namespace   |tableName             |isTemporary|
+------------+----------------------+-----------+
|ad_lakehouse|events                |false      |
|ad_lakehouse|maintenance_lab_events|false      |
+------------+----------------------+-----------+

[정상 데이터 확인]
+---------+-------------------+-------+-----+----------+----+----------+
|event_id |event_ts           |user_id|ad_id|event_type|cost|event_date|
+---------+-------------------+-------+-----+----------+----+----------+
|imp_1001 |2026-05-03 09:00:00|user_A |ad_10|impression|0.12|2026-05-03|
|clk_1001 |2026-05-03 09:03:00|user_A |ad_10|click     |0.0 |2026-05-03|
|conv_1001|2026-05-03 09:20:00|user_A |ad_10|conversion|0.0 |2026-05-03|
+---------+-------------------+-------+-----+----------+----+----------+

[현재 snapshot 목록]
+-----------------------+-------------------+---------+
|committed_at

In [7]:
# Cell 2. Rollback 실습: 장애 데이터 삽입 → rollback → 복구 확인
# 장애 전 snapshot 저장
before_error_snapshot_id = spark.sql(f"""
SELECT snapshot_id
FROM {FULL_TABLE}.snapshots
ORDER BY committed_at DESC
LIMIT 1
""").collect()[0]["snapshot_id"]

print("장애 전 snapshot_id =", before_error_snapshot_id)

# 장애 시나리오: 광고 비용이 비정상적으로 큰 데이터가 잘못 적재됨
spark.sql(f"""
INSERT INTO {FULL_TABLE} VALUES
('cost_spike_9001', TIMESTAMP '2026-05-03 10:00:00', 'user_Z', 'ad_999', 'impression', 999999.99, DATE '2026-05-03')
""")

print("[장애 데이터 확인: cost > 1000]")
spark.sql(f"""
SELECT *
FROM {FULL_TABLE}
WHERE cost > 1000
""").show(truncate=False)

print("[Rollback 실행 결과]")
spark.sql(f"""
CALL glue_catalog.system.rollback_to_snapshot(
  '{DB}.{TABLE}',
  {before_error_snapshot_id}
)
""").show(truncate=False)

print("[복구 확인: cost_spike_9001 조회 결과가 비어야 함]")
spark.sql(f"""
SELECT *
FROM {FULL_TABLE}
WHERE event_id = 'cost_spike_9001'
""").show(truncate=False)

print("[복구 후 전체 데이터]")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY event_ts").show(truncate=False)

장애 전 snapshot_id = 6135273341922304420
[장애 데이터 확인: cost > 1000]
+---------------+-------------------+-------+------+----------+---------+----------+
|event_id       |event_ts           |user_id|ad_id |event_type|cost     |event_date|
+---------------+-------------------+-------+------+----------+---------+----------+
|cost_spike_9001|2026-05-03 10:00:00|user_Z |ad_999|impression|999999.99|2026-05-03|
+---------------+-------------------+-------+------+----------+---------+----------+

[Rollback 실행 결과]
+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
|9075727736708199395 |6135273341922304420|
+--------------------+-------------------+

[복구 확인: cost_spike_9001 조회 결과가 비어야 함]
+--------+--------+-------+-----+----------+----+----------+
|event_id|event_ts|user_id|ad_id|event_type|cost|event_date|
+--------+--------+-------+-----+----------+----+----------+
+--------+--------+-------+-----+----------+----+-------

In [8]:
# Cell 3. Expire Snapshots: 전후 metadata/, data/ 크기 비교

from datetime import datetime, timedelta

# Iceberg procedure가 current_timestamp() 같은 함수식을 직접 받지 못하므로
# Python에서 cutoff 시간을 미리 계산한 뒤 TIMESTAMP 리터럴로 전달
expire_cutoff = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")

print("[Expire 전 metadata 크기]")
!aws s3 ls s3://s3-meta-iceberg-test/lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/metadata/ --recursive --human-readable --summarize

print("[Expire 전 data 크기]")
!aws s3 ls s3://s3-meta-iceberg-test/lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/data/ --recursive --human-readable --summarize

print("[expire_snapshots 실행 결과]")
print("expire_cutoff =", expire_cutoff)

spark.sql(f"""
CALL glue_catalog.system.expire_snapshots(
  table => '{DB}.{TABLE}',
  older_than => TIMESTAMP '{expire_cutoff}',
  retain_last => 1
)
""").show(truncate=False)

print("[Expire 후 metadata 크기]")
!aws s3 ls s3://s3-meta-iceberg-test/lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/metadata/ --recursive --human-readable --summarize

print("[Expire 후 data 크기]")
!aws s3 ls s3://s3-meta-iceberg-test/lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/data/ --recursive --human-readable --summarize

[Expire 전 metadata 크기]
2026-05-03 14:58:13    1.2 KiB lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/metadata/00000-3205a5f5-e20a-4c6d-b838-77622022e736.metadata.json
2026-05-03 15:47:54    1.2 KiB lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/metadata/00000-7fe42e9f-d69b-4a99-b1ef-124f432059b3.metadata.json
2026-05-03 15:21:05    1.2 KiB lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/metadata/00000-8ecc8533-477a-49cf-9b36-9c76791bd351.metadata.json
2026-05-03 15:53:28    1.2 KiB lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/metadata/00000-a8910579-158d-4d57-a874-720ff5c98d92.metadata.json
2026-05-03 14:37:12    1.2 KiB lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/metadata/00000-d1d641c7-9cab-45b6-9dcf-fc728cb1b05c.metadata.json
2026-05-03 15:15:14    1.2 KiB lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/metadata/00000-eda124c1-97bf-46ef-9d73-5f9b79c02e78.metadata.json
2026-05-03 15:48:

In [9]:
# Cell 4. Orphan File Cleanup 실습
# - Iceberg metadata가 모르는 파일을 S3 data/ 아래에 직접 업로드
# - SQL remove_orphan_files dry_run 시도
# - 현재 환경에서는 24시간 미만 interval 제한으로 즉시 cleanup이 막힘
# - 실습 후 AWS CLI로 수동 삭제하고 삭제 확인

import glob
from datetime import datetime, timedelta

# 1) Iceberg를 통하지 않은 Parquet 파일 생성
spark.createDataFrame(
    [("orphan_evt_01", "user_orphan", "ad_orphan", 777.77)],
    ["event_id", "user_id", "ad_id", "cost"]
).write.mode("overwrite").parquet("/tmp/orphan_lab_parquet")

print("[로컬 orphan parquet 생성 확인]")
!ls -lh /tmp/orphan_lab_parquet

# 2) S3 data/ 아래에 직접 업로드 → Iceberg metadata가 모르는 orphan file
print("[S3에 orphan 파일 직접 업로드]")

orphan_part_file = glob.glob("/tmp/orphan_lab_parquet/part-*.parquet")[0]
orphan_s3_file = "s3://s3-meta-iceberg-test/lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/data/manual_upload/orphan_lab_file.parquet"
orphan_s3_dir = "s3://s3-meta-iceberg-test/lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/data/manual_upload/"

print("업로드할 로컬 파일:", orphan_part_file)
print("업로드할 S3 경로:", orphan_s3_file)

!aws s3 cp {orphan_part_file} {orphan_s3_file}

print("[S3 업로드 확인: orphan 후보 파일]")
!aws s3 ls {orphan_s3_dir}

# 3) remove_orphan_files dry_run 시도
# SQL procedure는 방금 만든 파일을 즉시 지우는 24시간 미만 interval을 허용하지 않음
print("[remove_orphan_files dry_run 시도 결과]")

try:
    orphan_cutoff = (datetime.now() + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
    print("orphan_cutoff =", orphan_cutoff)

    spark.sql(f"""
    CALL glue_catalog.system.remove_orphan_files(
      table => '{DB}.{TABLE}',
      older_than => TIMESTAMP '{orphan_cutoff}',
      dry_run => true
    )
    """).show(truncate=False)

except Exception as e:
    print("dry_run 실행이 제한에 의해 실패했습니다.")
    print("실패 원인 요약:")
    print("Iceberg SQL remove_orphan_files procedure는 24시간보다 짧은 interval을 허용하지 않습니다.")
    print("방금 생성한 orphan file을 즉시 dry_run 대상으로 잡으려 하면 안전장치 때문에 막힙니다.")
    print("에러 메시지:")
    print(str(e)[:1200])

# 4) 실제 삭제는 AWS CLI로 수행
# 목적: 실습 후 S3에 남긴 수동 orphan 파일을 정리하고 삭제 확인까지 남김
print("[AWS CLI로 orphan 파일 수동 삭제]")
!aws s3 rm {orphan_s3_file}

print("[삭제 확인: 아무것도 안 나오면 삭제 완료]")
!aws s3 ls {orphan_s3_dir}

[Stage 59:>                                                         (0 + 2) / 2]

[로컬 orphan parquet 생성 확인]
total 8.0K
-rw-r--r-- 1 pi pi  565 May  3 15:53 part-00000-e7cccc15-51f1-4ba7-b889-87356c4c2d1f-c000.snappy.parquet
-rw-r--r-- 1 pi pi 1.4K May  3 15:53 part-00001-e7cccc15-51f1-4ba7-b889-87356c4c2d1f-c000.snappy.parquet
-rw-r--r-- 1 pi pi    0 May  3 15:53 _SUCCESS


[S3에 orphan 파일 직접 업로드]
업로드할 로컬 파일: /tmp/orphan_lab_parquet/part-00001-e7cccc15-51f1-4ba7-b889-87356c4c2d1f-c000.snappy.parquet
업로드할 S3 경로: s3://s3-meta-iceberg-test/lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/data/manual_upload/orphan_lab_file.parquet
upload: ../../tmp/orphan_lab_parquet/part-00001-e7cccc15-51f1-4ba7-b889-87356c4c2d1f-c000.snappy.parquet to s3://s3-meta-iceberg-test/lakehouse-lab/warehouse/ad_lakehouse.db/maintenance_lab_events/data/manual_upload/orphan_lab_file.parquet
[S3 업로드 확인: orphan 후보 파일]
2026-05-03 15:53:59       1357 orphan_lab_file.parquet
[remove_orphan_files dry_run 시도 결과]
orphan_cutoff = 2026-05-03 15:55:02
dry_run 실행이 제한에 의해 실패했습니다.
실패 원인 요약:
Iceberg SQL remove_orphan_files procedure는 24시간보다 짧은 interval을 허용하지 않습니다.
방금 생성한 orphan file을 즉시 dry_run 대상으로 잡으려 하면 안전장치 때문에 막힙니다.
에러 메시지:
Cannot remove orphan files with an interval less than 24 hours. Executing this procedure with a short interval may corrupt the table if other operations are ha

## Cell 5. 보고서용 정책 결정 및 근거

본 실습 테이블 `maintenance_lab_events`의 snapshot expire 정책은 다음과 같이 결정하였다.

```text
max-snapshot-age-ms = 259200000
min-snapshots-to-keep = 3
max-snapshot-age-ms = 259200000은 3일을 의미한다. maintenance_lab_events는 운영 테이블이 아니라 S3 기반 Iceberg 유지보수 기능을 검증하기 위한 실습용 테이블이므로 장기간 snapshot을 보존할 필요가 낮다. 다만 rollback 및 time travel 검증을 위해 최근 3일 이내의 snapshot은 유지하도록 설정하였다.

min-snapshots-to-keep = 3은 최소 3개의 snapshot을 항상 보존한다는 의미다. 실습 과정에서 정상 데이터 적재, 장애 데이터 적재, rollback 등 여러 단계의 변경이 발생하므로 최소 3개의 복구 지점을 유지하면 기본적인 장애 복구 검증이 가능하다.

본 실습에서는 expire_snapshots의 효과를 즉시 확인하기 위해 실습용 기준으로 snapshot 만료를 강하게 적용하였다. 실제 운영 환경에서는 최신 snapshot을 제외한 대부분의 snapshot이 한 번에 만료되지 않도록 데이터 적재 주기, 복구 요구 시간, 저장 비용을 함께 고려해 더 보수적인 정책을 적용해야 한다.

remove_orphan_files의 older_than은 운영 환경에서는 3~7일 이상으로 보수적으로 설정하는 것이 적절하다. 너무 짧은 기준을 사용하면 아직 진행 중인 write 작업의 임시 파일이나 지연 반영 중인 파일을 orphan file로 잘못 판단하여 테이블을 손상시킬 위험이 있다. 본 실습에서는 방금 수동 업로드한 orphan file을 즉시 정리하려고 시도했으나, Iceberg SQL procedure의 24시간 미만 interval 제한으로 즉시 cleanup은 차단되었다. 따라서 해당 제한 사항을 실습 결과에 기록하고, 수동으로 업로드한 실습용 orphan file은 AWS CLI를 통해 정리하였다.

현재 maintenance_lab_events는 특정 서비스 도메인에 종속된 운영 테이블이 아니라 Iceberg 유지보수 기능 검증을 위한 실습용 테이블이다. 추후 실제 도메인과 운영 요구사항이 확정되면 데이터 중요도, 적재 빈도, 장애 복구 목표, 저장 비용을 기준으로 snapshot 보존 기간과 최소 snapshot 개수를 별도로 재산정할 예정이다.
```